# NetObserver

Reads live status from a `Net` instance and returns Pydantic models.
All methods are synchronous (reading state snapshots).

In [ ]:
#|default_exp observe.observer

In [ ]:
#|export
from typing import Any

from netrun.net._net._net import Net
from netrun.net._net._context import EpochLog

from netrun_utils.observe.models import (
    NetStatus,
    NodeStatus,
    EdgeStatus,
    EpochInfo,
    StructuredLogEntry,
    LogEntry,
    ControlResponse,
)

In [ ]:
#|export
def _epoch_state_to_str(state) -> str:
    """Convert a netrun_sim EpochState enum to a string."""
    s = str(state).lower()
    if "startable" in s:
        return "startable"
    if "running" in s:
        return "running"
    if "finished" in s:
        return "finished"
    if "cancelled" in s:
        return "cancelled"
    return s


def _epoch_log_to_info(log: EpochLog) -> EpochInfo:
    """Convert an EpochLog dataclass to an EpochInfo model."""
    state = "finished"
    if log.outcome == "cancelled":
        state = "cancelled"
    structured_logs = [
        StructuredLogEntry(
            timestamp=entry.timestamp.isoformat(),
            message=entry.message,
            level=entry.level,
            fields=entry.fields,
        )
        for entry in (log.node_log_entries or [])
    ]

    return EpochInfo(
        epoch_id=log.epoch_id,
        node_name=log.node_name,
        state=state,
        created_at=log.created_at.isoformat(),
        started_at=log.started_at.isoformat() if log.started_at else None,
        ended_at=log.ended_at.isoformat() if log.ended_at else None,
        duration_ms=log.duration_ms,
        queue_time_ms=log.queue_time_ms,
        outcome=log.outcome,
        error=log.error,
        error_type=log.error_type,
        error_traceback=log.error_traceback,
        pool_id=log.pool_id,
        worker_id=log.worker_id,
        was_cache_hit=log.was_cache_hit,
        was_file_storage_hit=log.was_file_storage_hit,
        retry_count=log.retry_count,
        factory=log.factory,
        in_salvo_ports=log.in_salvo_ports,
        in_salvo_packet_count=log.in_salvo_packet_count,
        out_salvo_count=log.out_salvo_count,
        orphaned_packet_count=log.orphaned_packet_count,
        destroyed_packet_count=log.destroyed_packet_count,
        node_log_entries=structured_logs,
    )


def _epoch_state_to_info(epoch_state) -> EpochInfo:
    """Convert an _EpochState to an EpochInfo model."""
    return EpochInfo(
        epoch_id=epoch_state.id,
        node_name=epoch_state.node_name,
        state=_epoch_state_to_str(epoch_state.state),
        created_at=epoch_state.created_at.isoformat(),
        started_at=epoch_state.started_at.isoformat() if epoch_state.started_at else None,
        ended_at=epoch_state.ended_at.isoformat() if epoch_state.ended_at else None,
        duration_ms=None,
        outcome=None,
        pool_id=epoch_state.pool_id,
        worker_id=epoch_state.worker_id,
        was_cache_hit=epoch_state.was_cache_hit,
        was_file_storage_hit=epoch_state.was_file_storage_hit,
    )

## NetObserver

In [ ]:
#|export
class NetObserver:
    """Reads live status from a Net instance.

    Provides query methods for inspecting node status, edges, epochs, and logs,
    plus control methods for enabling/disabling nodes and injecting data.
    """

    def __init__(self, net: Net):
        self._net = net

    # --- Query methods ---

    def get_status(self) -> NetStatus:
        """Get overall net status."""
        resolved = self._net._config_resolved
        node_names = [nc.name for nc in resolved.graph.nodes]

        running_ids = set()
        for epoch_state in self._net.epochs.values():
            if _epoch_state_to_str(epoch_state.state) == "running":
                running_ids.add(epoch_state.node_name)

        busy = [name for name in node_names if name in running_ids]
        idle = [name for name in node_names if name not in running_ids]

        return NetStatus(
            started=self._net.started,
            paused=self._net.paused,
            is_blocked=self._net.is_blocked() if self._net.started else False,
            node_names=node_names,
            edge_count=len(self._net.graph.edges()),
            total_epochs=len(self._net.epochs),
            busy_nodes=busy,
            idle_nodes=idle,
            startable_epoch_count=len(self._net.get_startable_epochs()),
            running_epoch_count=len(running_ids),
            dead_letter_count=len(self._net.dead_letter_queue),
            exception_count=len(self._net.exception_queue),
            output_queues=self.get_output_queue_counts(),
        )

    def get_nodes(self) -> list[NodeStatus]:
        """Get status of all nodes."""
        resolved = self._net._config_resolved
        return [self._build_node_status(nc.name) for nc in resolved.graph.nodes]

    def get_node(self, name: str) -> NodeStatus:
        """Get status of a single node."""
        return self._build_node_status(name)

    def get_edges(self) -> list[EdgeStatus]:
        """Get status of all edges."""
        result = []
        for edge in self._net.graph.edges():
            # Count packets on this edge
            import netrun_sim
            location = netrun_sim.PacketLocation.edge(edge)
            packet_ids = self._net.netsim.get_packets_at_location(location)
            result.append(EdgeStatus(
                source_node=edge.source.node_name,
                source_port=edge.source.port_name,
                target_node=edge.target.node_name,
                target_port=edge.target.port_name,
                packet_count=len(packet_ids),
            ))
        return result

    def get_epoch_logs(self) -> list[EpochInfo]:
        """Get all epoch information.

        Combines retained EpochLog objects (for completed epochs) with
        live _EpochState objects (for running/startable epochs).
        """
        result = []

        # Completed epochs from retained logs
        for log in self._net.epoch_logs.values():
            result.append(_epoch_log_to_info(log))

        # Running and startable epochs from live state
        completed_ids = {info.epoch_id for info in result}
        for epoch_state in self._net.epochs.values():
            if epoch_state.id not in completed_ids:
                result.append(_epoch_state_to_info(epoch_state))

        return result

    def get_all_logs(self) -> list[LogEntry]:
        """Get all logs in chronological order."""
        return [
            LogEntry(
                timestamp=ts.isoformat(),
                message=message,
                node_name=node_name,
                epoch_id=str(epoch_id),
            )
            for ts, epoch_id, node_name, message in self._net.logs.all_chronological()
        ]

    def get_node_logs(self, node_name: str) -> list[LogEntry]:
        """Get logs for a specific node."""
        return [
            LogEntry(
                timestamp=ts.isoformat(),
                message=message,
                node_name=node_name,
            )
            for ts, message in self._net.logs.for_node(node_name)
        ]

    # --- Control methods ---

    def enable_node(self, name: str) -> ControlResponse:
        """Enable a node."""
        try:
            self._net.enable_node(name)
            return ControlResponse(ok=True, message=f"Node '{name}' enabled")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    def disable_node(self, name: str) -> ControlResponse:
        """Disable a node."""
        try:
            self._net.disable_node(name)
            return ControlResponse(ok=True, message=f"Node '{name}' disabled")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    def send_control(self, node_name: str, control_type: str, value: Any = None) -> ControlResponse:
        """Send a control signal to a node."""
        try:
            self._net.send_control(node_name, control_type, value)
            return ControlResponse(ok=True, message=f"Control '{control_type}' sent to '{node_name}'")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    async def pause(self) -> ControlResponse:
        """Pause the net."""
        try:
            await self._net.pause()
            return ControlResponse(ok=True, message="Net paused")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    async def resume(self) -> ControlResponse:
        """Resume the net."""
        try:
            await self._net.resume()
            return ControlResponse(ok=True, message="Net resumed")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    def inject_data(self, node_name: str, port_name: str, values: list[Any]) -> ControlResponse:
        """Inject data into a node's input port."""
        try:
            packet_ids = self._net.inject_data(node_name, port_name, values)
            return ControlResponse(ok=True, message=f"Injected {len(packet_ids)} packets into '{node_name}.{port_name}'")
        except Exception as e:
            return ControlResponse(ok=False, message=str(e))

    # --- Net-level data ---

    def get_dead_letter_queue(self) -> list[dict]:
        """Get dead letter queue entries."""
        result = []
        for entry in self._net.dead_letter_queue:
            result.append({
                "epoch_id": str(entry.get("epoch_id", "")),
                "node_name": entry.get("node_name", ""),
                "error": str(entry.get("error", "")),
                "error_type": type(entry.get("error")).__name__ if entry.get("error") else None,
                "retry_count": entry.get("retry_count", 0),
            })
        return result

    def get_output_queue_counts(self) -> dict[str, int]:
        """Get output queue names and their current packet counts."""
        result = {}
        resolved = self._net._config_resolved
        if resolved.output_queues:
            for queue_name in resolved.output_queues:
                queue = self._net._output_queues.get(queue_name)
                result[queue_name] = queue.qsize() if queue else 0
        return result

    # --- Config ---

    def get_config(self) -> dict:
        """Get the resolved net config as a JSON-serializable dict."""
        return self._net._config_resolved.model_dump(mode="json")

    # --- Private helpers ---

    def _build_node_status(self, node_name: str) -> NodeStatus:
        """Build a NodeStatus model for a node by name."""
        node_config = self._net.get_node_config(node_name)

        # Enabled state
        enabled = self._net.is_node_enabled(node_name)

        # Epoch counts and IDs
        running_ids = []
        startable_ids = []
        epoch_count = 0
        for epoch_state in self._net.epochs.values():
            if epoch_state.node_name == node_name:
                epoch_count += 1
                state_str = _epoch_state_to_str(epoch_state.state)
                if state_str == "running":
                    running_ids.append(epoch_state.id)
                elif state_str == "startable":
                    startable_ids.append(epoch_state.id)

        # Input port packet counts
        port_packets = {}
        for port_name, packets in self._net.get_packets_at_all_ports(node_name).items():
            port_packets[port_name] = len(packets)

        # Port names
        in_port_names = list(node_config.in_ports.keys()) if node_config.in_ports else []
        out_port_names = list(node_config.out_ports.keys()) if node_config.out_ports else []

        # Pools
        pools = []
        if node_config.execution_config and node_config.execution_config.pools:
            pools = list(node_config.execution_config.pools)

        # Factory
        factory = str(node_config.factory) if node_config.factory else None

        return NodeStatus(
            name=node_name,
            enabled=enabled,
            epoch_count=epoch_count,
            is_busy=len(running_ids) > 0,
            running_epoch_ids=running_ids,
            startable_epoch_ids=startable_ids,
            in_port_names=in_port_names,
            out_port_names=out_port_names,
            input_port_packet_counts=port_packets,
            pools=pools,
            factory=factory,
        )